# NeurIPS 2026 Smart Buildings — Version 25: CBF-QP safety filtering

**Hypothesis:** Keep Version 7's energy-efficient thermodynamic dispatch and reduce severe zone-temperature excursions using a lightweight, soft discrete-time Control Barrier Function quadratic program.

## Architecture

- Nominal controller: unchanged Version 7 physics policy, except for renaming the class. It still controls equipment on/off, coil, economizer, VAVs, and hot water.
- Safety layer: filtered temperature rate, upper/lower comfort barriers, per-zone VAV/reheat soft-QPs, and an optional fan-pressure soft-QP for severe overheating.
- Deterministic runtime: 26 bisection steps per active QP; O(AHUs + zones) for a fixed solve tolerance; only NumPy and official Kaggle gateway.
- Bounded intervention: at most +0.25 zone damper/reheat opening fraction and +0.18 normalized fan-pressure action during severe excursions; at most +0.08 VAV opening fraction near a comfort boundary. Equipment on/off and cooling coil settings remain exactly V7's.

## Barrier constraints

Let T be zone temperature, L/U the applicable heating/cooling setpoints, m=0.20 K the guard inset, and alpha=0.28. Require approximately h(T_next) >= (1-alpha) h(T), for h_upper=U-m-T and h_lower=T-L-m.

Each QP minimizes 0.5 * sum(w_i * correction_i^2) + 0.5 * penalty * slack^2 subject to sum(thermal_gain_i * correction_i) + slack >= estimated_barrier_deficit, with box-bounded nonnegative corrections and nonnegative slack. Dual bisection solves the diagonal QP. Soft slack handles actuator saturation.

**Limitations:** The gains are engineering proxies, not calibrated or robustly bounded plant sensitivities. This is *not* a formal safety guarantee. Version 7 outdoor-air handling is preserved deliberately, including any existing approximation, so the isolated change is the filter. Confirm actual energy use and comfort excursions in the organizer simulator.

## Validation

Run: python Co_Kaggle/g8/submissions/test_version25.py

Submit submission-nb - Version 25.ipynb as a Kaggle code notebook. Compare it against Version 7 public loss 0.03947, plus frequency/duration of temperature excursions beyond 1 K and 2 K, comfort degree-minutes, energy, and inference latency. **Version 25 is unscored.**


In [ ]:
"""Version 7: physics-based, load-matched HVAC dispatch.

Distinct experiment: use the AHU mixing equation and estimated delivered
supply-air temperature to size airflow and avoid unnecessary compressor work.
This is feed-forward engineering control, not PID, temperature prediction,
risk-weighted allocation, RLS, or candidate-action optimization.

Runtime: O(number of AHUs + number of zones) per five-minute step.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class PhysicsBaselinePolicy(smart_buildings_inference_server.Policy):
    """Weather-compensated, thermodynamics-aware air and hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    ACTIVE_THRESHOLD_K = 0.18
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """No episode state: decisions depend only on the current observation."""
        return None

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.ACTIVE_THRESHOLD_K:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.ACTIVE_THRESHOLD_K
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using actual heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.ACTIVE_THRESHOLD_K:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + 2.1 * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.ACTIVE_THRESHOLD_K:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + 0.31 * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.ACTIVE_THRESHOLD_K:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Read zones -> plan AHU air -> set VAVs -> coordinate shared water.

        Performs one pass per AHU and zone; no search, learned model, or
        state carried between five-minute decisions.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = [
                self._read_zone(obs, z, fallback) for z in ahu["zones"]
            ]
            statuses = [s for s in statuses if s is not None]
            outdoor = self._value(obs, ahu["outdoor"])
            if np.isfinite(outdoor) and 210.0 <= outdoor <= 330.0:
                outdoors.append(outdoor)
            else:
                outdoor = float("nan")
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


class TransferPolicy(PhysicsBaselinePolicy):
    """Soft discrete-time CBF-QP safety filter around the unchanged V7 policy.

    V7 owns equipment switching, economizer dispatch, and coil setpoints.
    This filter only makes bounded one-sided corrections to active VAVs
    and, for severe overheating, the associated AHU fan pressure.

    Thermal actuator gains are proxies, not a calibrated building model.
    The QP mitigates predicted risk but does NOT guarantee actual safety.
    """

    BARRIER_ALPHA = 0.28
    GUARD_BAND_K = 0.20
    NEAR_BOUNDARY_K = 0.40
    SEVERE_EXCURSION_K = 0.55

    MAX_DAMPER_CORRECTION = 0.25
    MAX_VALVE_CORRECTION = 0.25
    MAX_FAN_ACTION_CORRECTION = 0.18
    MAX_RATE_K_PER_STEP = 0.30
    MAX_VALID_STEP_K = 1.50
    QP_BISECTION_STEPS = 26

    def reset(self):
        """Discard all temperature history at the episode boundary."""
        super().reset()
        self._zone_history = {}

    @staticmethod
    def _valid_index(index, length):
        return isinstance(index, (int, np.integer)) and 0 <= index < length

    def _zone_bounds(self, obs, zone, fallback):
        """Match V7 setpoint validation, including its schedule fallback."""
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            return fallback
        return low, high

    def _temperature_rate(self, key, temperature):
        """Smooth observed K/step change; reject discontinuous sensor jumps."""
        previous = self._zone_history.get(key)
        rate = 0.0
        if previous is not None:
            difference = temperature - previous[0]
            if abs(difference) <= self.MAX_VALID_STEP_K:
                rate = (0.45 * previous[1]
                        + 0.55 * self._clamp(
                            difference,
                            -self.MAX_RATE_K_PER_STEP,
                            self.MAX_RATE_K_PER_STEP,
                        ))
        self._zone_history[key] = (temperature, rate)
        return rate

    @classmethod
    def _solve_soft_cbf_qp(cls, deficit, gains, caps, weights, penalty):
        """Solve a diagonal, box-constrained soft CBF quadratic program.

        min 0.5 * sum(weight_i * delta_i**2) + 0.5 * penalty * slack**2
        s.t. sum(gain_i * delta_i) + slack >= deficit
             0 <= delta_i <= cap_i, slack >= 0

        KKT gives delta_i = clip(lambda * gain_i / weight_i, 0, cap_i)
        and slack = lambda / penalty. A fixed-count dual bisection
        avoids a dependency on scipy or an external QP solver.
        """
        if not (len(gains) == len(caps) == len(weights)):
            raise ValueError("QP actuator arrays must have equal length")
        if deficit <= 0:
            return [0.0 for _ in gains], 0.0
        lower_lambda = 0.0
        upper_lambda = penalty * deficit
        for _ in range(cls.QP_BISECTION_STEPS):
            multiplier = 0.5 * (lower_lambda + upper_lambda)
            improvement = multiplier / penalty
            for gain, cap, weight in zip(gains, caps, weights):
                improvement += gain * min(cap, multiplier * gain / weight)
            if improvement < deficit:
                lower_lambda = multiplier
            else:
                upper_lambda = multiplier
        deltas = [
            min(cap, upper_lambda * gain / weight)
            for gain, cap, weight in zip(gains, caps, weights)
        ]
        return deltas, upper_lambda / penalty

    def _filter_zone(self, obs, zone, ahu, action, fallback):
        """Filter an active zone; return residual upper-bound risk in K."""
        status = self._read_zone(obs, zone, fallback)
        key = zone["temperature"]
        if status is None:
            self._zone_history.pop(key, None)
            return 0.0, 0.0

        temperature = status["temp"]
        low, high = self._zone_bounds(obs, zone, fallback)
        rate = self._temperature_rate(key, temperature)
        continuous = action["continuous_action"]
        discrete = action["discrete_action"]
        if discrete[ahu["run"]] != 1:
            # V7 remains responsible for starting and stopping equipment.
            return 0.0, 0.0

        # h_upper = high - guard - T; h_lower = T - low - guard.
        # With h_next >= (1-alpha)*h_now and one-step nominal
        # prediction T_next = T + rate, each barrier becomes a QP
        # demand in Kelvin of corrective predicted thermal response.
        upper_deficit = rate + self.BARRIER_ALPHA * (
            temperature - (high - self.GUARD_BAND_K))
        lower_deficit = self.BARRIER_ALPHA * (
            (low + self.GUARD_BAND_K) - temperature) - rate
        damper_index = zone["damper"]
        valve_index = zone["valve"]

        if (upper_deficit > 0.0
                and temperature >= high - self.NEAR_BOUNDARY_K
                and upper_deficit >= lower_deficit):
            if not self._valid_index(damper_index, len(continuous)):
                return upper_deficit, max(0.0, temperature - high)
            fraction = 0.5 * (float(continuous[damper_index]) + 1.0)
            supply_k = 295.0 + 10.0 * float(continuous[ahu["supply"]])
            cooling_gain = self._clamp(
                0.011 * max(0.0, temperature - supply_k), 0.0, 0.17)
            severe = temperature - high >= self.SEVERE_EXCURSION_K
            cap = min(
                self.MAX_DAMPER_CORRECTION if severe else 0.08,
                max(0.0, 0.98 - fraction),
            )
            corrections, _ = self._solve_soft_cbf_qp(
                upper_deficit, [cooling_gain], [cap], [1.0],
                16.0 if severe else 4.0)
            correction = corrections[0]
            continuous[damper_index] = self._fraction_action(
                fraction + correction)
            residual = max(0.0, upper_deficit - cooling_gain * correction)
            return residual, max(0.0, temperature - high)

        if (lower_deficit > 0.0
                and temperature <= low + self.NEAR_BOUNDARY_K
                and 0 <= self.water_run < len(discrete)
                and discrete[self.water_run] == 1):
            # Only change reheat if V7 already enabled the hot-water circuit.
            gains, caps, weights, indices, fractions = [], [], [], [], []
            severe = low - temperature >= self.SEVERE_EXCURSION_K
            damper_cap = self.MAX_DAMPER_CORRECTION if severe else 0.08
            valve_cap = self.MAX_VALVE_CORRECTION if severe else 0.08
            for index, gain, weight, maximum in (
                (damper_index, 0.045, 2.0, damper_cap),
                (valve_index, 0.125, 1.0, valve_cap),
            ):
                if self._valid_index(index, len(continuous)):
                    fraction = 0.5 * (float(continuous[index]) + 1.0)
                    indices.append(index)
                    fractions.append(fraction)
                    gains.append(gain)
                    weights.append(weight)
                    caps.append(min(maximum, max(0.0, 0.98 - fraction)))
            if gains:
                corrections, _ = self._solve_soft_cbf_qp(
                    lower_deficit, gains, caps, weights,
                    16.0 if severe else 4.0)
                for index, fraction, correction in zip(
                        indices, fractions, corrections):
                    continuous[index] = self._fraction_action(
                        fraction + correction)

        return 0.0, 0.0

    def get_action(self, obs):
        """Run V7 nominal control, then the minimal safety filter."""
        action = super().get_action(obs)
        observation = np.asarray(obs, dtype=np.float64).reshape(-1)
        fallback = self._schedule_fallback(observation)
        continuous = action["continuous_action"]
        discrete = action["discrete_action"]

        for ahu in self.ahus:
            worst_residual = 0.0
            worst_excursion = 0.0
            for zone in ahu["zones"]:
                residual, excursion = self._filter_zone(
                    observation, zone, ahu, action, fallback)
                worst_residual = max(worst_residual, residual)
                worst_excursion = max(worst_excursion, excursion)

            # Avoid fan-power overhead except for severe overheating.
            pressure = ahu["pressure"]
            if (discrete[ahu["run"]] == 1
                    and worst_excursion >= self.SEVERE_EXCURSION_K
                    and worst_residual > 0.0
                    and self._valid_index(pressure, len(continuous))):
                command = float(continuous[pressure])
                cap = min(self.MAX_FAN_ACTION_CORRECTION,
                          max(0.0, 1.0 - command))
                corrections, _ = self._solve_soft_cbf_qp(
                    worst_residual, [0.065], [cap], [4.0], 12.0)
                continuous[pressure] = self._clamp(
                    command + corrections[0], -1.0, 1.0)

        # Preserve the original Kaggle float32/int32 action contract.
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return action


In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
